# Phase 2: Data Preprocessing Pipeline (Training Set Only)
## CICIoT2023 Network Intrusion Detection
### Đề tài Tốt nghiệp: Hệ thống phát hiện tấn công mạng IoT dựa trên XAI

---

### 🎯 Mục tiêu & Nguyên tắc Thiết kế Pipeline Tiền Xử Lý
1. **Chống Rò rỉ Dữ liệu (Strict Leakage Prevention)**:
   - Toàn bộ tham số tiền xử lý (`RobustScaler`, `imputation_medians`, `LabelEncoder`, `variance_threshold`) chỉ được fit **duy nhất trên tập huấn luyện (`train`)**.
   - Đóng gói toàn bộ pipeline vào `models/preprocessor.joblib` để áp dụng bất biến lên tập validation/test sau này.
2. **Khử Trùng Lặp (Deduplication - Task 2.1)**:
   - Loại bỏ các dòng luồng trùng lặp chính xác (`drop_duplicates()`), tránh làm thổi phồng chỉ số accuracy ảo và rò rỉ chéo giữa train và test.
3. **Lấy Mẫu Phân Tầng Không Lệch (Stratified Sampling - Không dùng `nrows`)**:
   - Khi chạy thử nghiệm trên tập con, luôn dùng `train_test_split(stratify=y)` để bảo toàn 100% tỷ lệ các loại tấn công.
4. **Kỹ thuật Trích Xuất Đặc Trưng (Feature Engineering - Task 2.3)**:
   - Xây dựng 14 đặc trưng miền an ninh mạng (tỷ lệ gói, cờ TCP, luồng giao thức).
   - Nén các giá trị cực trị của phép chia bằng hàm $\log(1 + x)$ (`np.log1p`) để bảo vệ khoảng cách Euclidean trong thuật toán $k$-NN của SMOTE và mô hình mạng nơ-ron.
5. **Phân Cấp Nhãn Đa Tầng (Task 2.4)**:
   - Cấp 1: **34 Lớp Chi tiết** (33 cuộc tấn công + BenignTraffic).
   - Cấp 2: **9 Nhóm Tấn công** (DDoS, DoS, Mirai, Recon, Spoofing, Web, BruteForce, Malware, Benign).
   - Cấp 3: **Nhị phân** (0: Benign, 1: Attack).

In [ ]:
# ==============================================================================
# Cell 1: Khởi tạo Môi trường & Định nghĩa Schema Dữ liệu
# ==============================================================================
import os
import gc
import json
import warnings
import numpy as np
import pandas as pd
import joblib

from sklearn.preprocessing import LabelEncoder, RobustScaler
from sklearn.model_selection import train_test_split

warnings.filterwarnings('ignore')

os.makedirs('data', exist_ok=True)
os.makedirs('models', exist_ok=True)

# Mapping 33 tấn công + Benign -> 9 nhóm
ATTACK_CATEGORY_MAPPING = {
    'DDoS-ICMP_Flood': 'DDoS', 'DDoS-UDP_Flood': 'DDoS', 'DDoS-TCP_Flood': 'DDoS',
    'DDoS-PSHACK_Flood': 'DDoS', 'DDoS-SYN_Flood': 'DDoS', 'DDoS-RSTFINFlood': 'DDoS',
    'DDoS-SynonymousIP_Flood': 'DDoS', 'DDoS-ICMP_Fragmentation': 'DDoS',
    'DDoS-UDP_Fragmentation': 'DDoS', 'DDoS-ACK_Fragmentation': 'DDoS',
    'DDoS-HTTP_Flood': 'DDoS', 'DDoS-SlowLoris': 'DDoS',
    'DoS-UDP_Flood': 'DoS', 'DoS-TCP_Flood': 'DoS', 'DoS-SYN_Flood': 'DoS', 'DoS-HTTP_Flood': 'DoS',
    'Mirai-greeth_flood': 'Mirai', 'Mirai-udpplain': 'Mirai', 'Mirai-greip_flood': 'Mirai',
    'Recon-HostDiscovery': 'Recon', 'Recon-OSScan': 'Recon', 'Recon-PortScan': 'Recon',
    'Recon-PingSweep': 'Recon', 'VulnerabilityScan': 'Recon',
    'MITM-ArpSpoofing': 'Spoofing', 'DNS_Spoofing': 'Spoofing',
    'BrowserHijacking': 'Web', 'CommandInjection': 'Web', 'SqlInjection': 'Web',
    'XSS': 'Web', 'Uploading_Attack': 'Web',
    'DictionaryBruteForce': 'BruteForce',
    'Backdoor_Malware': 'Malware',
    'BenignTraffic': 'Benign'
}

DTYPE_DICT = {
    'flow_duration': 'float32', 'Header_Length': 'float32', 'Protocol Type': 'float32',
    'Duration': 'float32', 'Rate': 'float32', 'Srate': 'float32', 'Drate': 'float32',
    'fin_flag_number': 'float32', 'syn_flag_number': 'float32', 'rst_flag_number': 'float32',
    'psh_flag_number': 'float32', 'ack_flag_number': 'float32', 'ece_flag_number': 'float32',
    'cwr_flag_number': 'float32', 'ack_count': 'float32', 'syn_count': 'float32',
    'fin_count': 'float32', 'urg_count': 'float32', 'rst_count': 'float32',
    'HTTP': 'float32', 'HTTPS': 'float32', 'DNS': 'float32', 'Telnet': 'float32',
    'SMTP': 'float32', 'SSH': 'float32', 'IRC': 'float32', 'TCP': 'float32',
    'UDP': 'float32', 'DHCP': 'float32', 'ARP': 'float32', 'ICMP': 'float32',
    'IPv': 'float32', 'LLC': 'float32', 'Tot sum': 'float32', 'Min': 'float32',
    'Max': 'float32', 'AVG': 'float32', 'Std': 'float32', 'Tot size': 'float32',
    'IAT': 'float32', 'Number': 'float32', 'Magnitue': 'float32', 'Radius': 'float32',
    'Covariance': 'float32', 'Variance': 'float32', 'Weight': 'float32',
    'label': 'object'
}
print(f"Đã định nghĩa 46 đặc trưng và mapping cho {len(ATTACK_CATEGORY_MAPPING)} nhãn.")

## Bước 1: Tải Dữ liệu Huấn Luyện (Training Set) với Lấy Mẫu Phân Tầng
> 🔴 **Lưu ý Quan trọng**: Không dùng `nrows` để lấy mẫu vì file CSV CICIoT2023 được lưu tuần tự theo từng cuộc tấn công. Phải dùng **Stratified Sampling** để đảm bảo đầy đủ đại diện mọi loại tấn công.

In [ ]:
# ==============================================================================
# Cell 2: Tải Dữ liệu Huấn luyện & Lấy Mẫu Phân tầng (Nếu Cần)
# ==============================================================================
candidate_paths = [
    'data/X_train.parquet',
    'train.csv',
    'plots/train.csv',
    '../train.csv'
]

data_path = None
for p in candidate_paths:
    if os.path.exists(p):
        data_path = p
        break

print(f"Nguồn dữ liệu huấn luyện: {data_path}")
if data_path and data_path.endswith('.parquet'):
    df = pd.read_parquet(data_path)
    # Nạp y_train tương ứng nếu nạp parquet
    y_t = joblib.load('data/y_train.pkl')
    df['label'] = y_t['label_name']
elif data_path:
    df = pd.read_csv(data_path, dtype=DTYPE_DICT)
else:
    # Fallback to pre-loaded X_train.pkl
    df = joblib.load('data/X_train.pkl')
    y_t = joblib.load('data/y_train.pkl')
    df['label'] = y_t['label_name']

print(f"Kích thước dữ liệu nạp: {df.shape[0]:,} dòng x {df.shape[1]} cột")

## Bước 2: Khử Trùng Lặp & Làm Sạch Missing / Infinite Values (Task 2.1)
1. **Khử trùng lặp (`drop_duplicates`)**: Loại bỏ các dòng luồng trùng lặp 100% trong tập huấn luyện.
2. **Xử lý Inf / NaN**: Chuyển Inf thành NaN, sau đó tính **Median** của từng cột trên tập train và điền khuyết.

In [ ]:
# ==============================================================================
# Cell 3: Khử Trùng Lặp & Điền Khuyết Bằng Median
# ==============================================================================
print("="*70)
print("TASK 2.1: KHỬ TRÙNG LẶP & LÀM SẠCH GIÁ TRỊ KHUYẾT / VÔ HẠN")
print("="*70)

# 1. Khử trùng lặp
n_dup = df.duplicated().sum()
if n_dup > 0:
    print(f"Phát hiện {n_dup:,} dòng trùng lặp ({n_dup / len(df) * 100:.2f}%). Đang loại bỏ...")
    df.drop_duplicates(inplace=True)
    df.reset_index(drop=True, inplace=True)
    print(f"-> Số dòng duy nhất còn lại: {len(df):,}")
else:
    print("Không có dòng trùng lặp trong tập dữ liệu.")

# 2. Xử lý giá trị vô hạn
numeric_cols = df.select_dtypes(include=['float32', 'float64']).columns.tolist()
inf_count = np.isinf(df[numeric_cols].values).sum()
if inf_count > 0:
    print(f"Phát hiện {inf_count:,} giá trị inf -> Thay bằng NaN")
    df.replace([np.inf, -np.inf], np.nan, inplace=True)

# 3. Tính toán Median trên tập Train
imputation_medians = {col: float(df[col].median()) for col in numeric_cols}
for col in numeric_cols:
    if df[col].isnull().any():
        df[col] = df[col].fillna(imputation_medians[col])

if 'label' in df.columns and df['label'].isnull().sum() > 0:
    df.dropna(subset=['label'], inplace=True)

assert df.isnull().sum().sum() == 0, "Vẫn còn giá trị NaN!"
print("Tập dữ liệu sạch 100% missing và infinite values.")

## Bước 3: Loại Bỏ Đặc Trưng Hằng Số (Task 2.2)
Loại bỏ các đặc trưng có phương sai $\le 10^{-12}$ (không mang thông tin phân biệt).

In [ ]:
# ==============================================================================
# Cell 4: Loại Bỏ Đặc Trưng Phương Sai Bằng Không (Task 2.2)
# ==============================================================================
feat_cols = [c for c in df.columns if c != 'label']
vars_ = df[feat_cols].var()
dropped_zero_var_cols = vars_[vars_ <= 1e-12].index.tolist()

if dropped_zero_var_cols:
    print(f"Loại bỏ {len(dropped_zero_var_cols)} đặc trưng hằng số: {dropped_zero_var_cols}")
    df.drop(columns=dropped_zero_var_cols, inplace=True)
else:
    print("Không có đặc trưng hằng số nào.")

## Bước 4: Centralized Feature Engineering (Task 2.3)
- **Trích xuất 14 đặc trưng miền an ninh mạng**: Tỷ lệ kích thước gói, tỉ lệ cờ SYN/ACK, tổng cờ TCP, nhóm giao thức mạng.
- **Nén Outlier Bằng Log1p**: Áp dụng $\log(1 + x)$ trên các tỷ lệ có mẫu số nhỏ để tránh bùng nổ giá trị $10^6 - 10^9$, ổn định khoảng cách Euclidean cho SMOTE và mô hình Neural Network.

In [ ]:
# ==============================================================================
# Cell 5: Trích Xuất Đặc Trưng Mới & Nén Tỷ Lệ (Task 2.3)
# ==============================================================================
def engineer_features(data, medians=None):
    eps = 1e-6
    
    # 1. Ratios (Nén bằng log1p)
    if 'Tot sum' in data.columns and 'Number' in data.columns:
        data['avg_packet_size'] = np.log1p(np.clip(data['Tot sum'] / (data['Number'] + eps), 0, 1e6)).astype('float32')
    if 'Srate' in data.columns and 'Rate' in data.columns:
        data['rate_ratio'] = np.log1p(np.clip(data['Srate'] / (data['Rate'] + eps), 0, 1e6)).astype('float32')
    if 'Drate' in data.columns and 'Rate' in data.columns:
        data['drate_ratio'] = np.log1p(np.clip(data['Drate'] / (data['Rate'] + eps), 0, 1e6)).astype('float32')
    if 'Duration' in data.columns and 'flow_duration' in data.columns:
        data['duration_ratio'] = np.log1p(np.clip(data['Duration'] / (data['flow_duration'] + eps), 0, 1e6)).astype('float32')
    if 'Header_Length' in data.columns and 'Tot size' in data.columns:
        data['header_to_payload_ratio'] = np.log1p(np.clip(data['Header_Length'] / (data['Tot size'] + eps), 0, 1e6)).astype('float32')

    # 2. Flag Aggregates
    flag_nums = [c for c in ['fin_flag_number', 'syn_flag_number', 'rst_flag_number', 
                             'psh_flag_number', 'ack_flag_number', 'ece_flag_number', 'cwr_flag_number'] if c in data.columns]
    if flag_nums:
        data['total_flags'] = data[flag_nums].sum(axis=1).astype('float32')

    flag_counts = [c for c in ['ack_count', 'syn_count', 'fin_count', 'urg_count', 'rst_count'] if c in data.columns]
    if flag_counts:
        data['total_counts'] = data[flag_counts].sum(axis=1).astype('float32')

    if 'syn_count' in data.columns and 'ack_count' in data.columns:
        data['syn_ack_ratio'] = np.log1p(np.clip(data['syn_count'] / (data['ack_count'] + eps), 0, 1e6)).astype('float32')
    if 'rst_count' in data.columns and 'syn_count' in data.columns:
        data['rst_syn_ratio'] = np.log1p(np.clip(data['rst_count'] / (data['syn_count'] + eps), 0, 1e6)).astype('float32')

    # 3. Protocols
    if 'HTTP' in data.columns and 'HTTPS' in data.columns:
        data['web_traffic'] = (data['HTTP'] + data['HTTPS']).astype('float32')
    trans = [c for c in ['TCP', 'UDP', 'ICMP'] if c in data.columns]
    if trans:
        data['transport_traffic'] = data[trans].sum(axis=1).astype('float32')
    mgmt = [c for c in ['DNS', 'DHCP', 'ARP'] if c in data.columns]
    if mgmt:
        data['network_mgmt'] = data[mgmt].sum(axis=1).astype('float32')
    remote = [c for c in ['Telnet', 'SSH'] if c in data.columns]
    if remote:
        data['remote_access'] = data[remote].sum(axis=1).astype('float32')
    if 'HTTPS' in data.columns and 'SSH' in data.columns:
        data['is_encrypted'] = (data['HTTPS'] + data['SSH']).astype('float32')

    data.replace([np.inf, -np.inf], np.nan, inplace=True)
    if medians:
        for c, med in medians.items():
            if c in data.columns and data[c].isnull().any():
                data[c] = data[c].fillna(med)
    data.fillna(0, inplace=True)
    return data

df = engineer_features(df, medians=imputation_medians)
print(f"Đã tạo thành công các đặc trưng mới. Kích thước hiện tại: {df.shape[1]} cột.")

## Bước 5: Phân Cấp Nhãn Tấn Công & Chuẩn Hóa RobustScaler (Task 2.4 & 2.5)

In [ ]:
# ==============================================================================
# Cell 6: Phân cấp Nhãn & Chuẩn Hóa RobustScaler
# ==============================================================================
df['attack_category'] = df['label'].map(lambda x: ATTACK_CATEGORY_MAPPING.get(x, 'Other'))
df['is_attack'] = (df['label'] != 'BenignTraffic').astype('int8')

targets = ['label', 'attack_category', 'is_attack']
features = [c for c in df.columns if c not in targets]

X_train = df[features].copy()
y_train = df['label'].copy()
y_cat_train = df['attack_category'].copy()
y_bin_train = df['is_attack'].copy()

# Encode
le_label = LabelEncoder()
y_train_enc = le_label.fit_transform(y_train)

le_cat = LabelEncoder()
y_cat_train_enc = le_cat.fit_transform(y_cat_train)

label_mapping = {label: int(code) for code, label in enumerate(le_label.classes_)}
cat_mapping = {cat: int(code) for code, cat in enumerate(le_cat.classes_)}

# Robust Scaling
print("Fit RobustScaler trên tập train...")
scaler = RobustScaler()
X_train_scaled = pd.DataFrame(
    scaler.fit_transform(X_train),
    columns=X_train.columns,
    index=X_train.index,
    dtype='float32'
)

y_train_dict = {
    'label_encoded': y_train_enc,
    'label_name': y_train.values,
    'category_encoded': y_cat_train_enc,
    'category_name': y_cat_train.values,
    'is_attack': y_bin_train.values
}
print(f"Chuẩn hóa hoàn tất! X_train: {X_train_scaled.shape[0]:,} dòng x {X_train_scaled.shape[1]} đặc trưng.")

## Bước 6: Lưu Deliverables & Pipeline Tiền Xử Lý
Lưu trữ các artifacts vào `data/` và `models/`.

In [ ]:
# ==============================================================================
# Cell 7: Lưu Trữ Deliverables
# ==============================================================================
# 1. Save Datasets
joblib.dump(X_train_scaled, 'data/X_train.pkl', compress=3)
joblib.dump(y_train_dict, 'data/y_train.pkl', compress=3)
print("Đã lưu: data/X_train.pkl & data/y_train.pkl")

# 2. Save Preprocessor Pipeline
preprocessor = {
    'scaler': scaler,
    'feature_names': X_train_scaled.columns.tolist(),
    'dropped_zero_var_cols': dropped_zero_var_cols,
    'imputation_medians': imputation_medians,
    'label_encoder': le_label,
    'category_encoder': le_cat,
    'label_mapping': label_mapping,
    'category_mapping': cat_mapping,
    'attack_category_mapping': ATTACK_CATEGORY_MAPPING,
    'engineered_features': [c for c in X_train_scaled.columns if c not in features or c not in DTYPE_DICT]
}
joblib.dump(preprocessor, 'models/preprocessor.joblib')

with open('models/label_mapping.json', 'w') as f:
    json.dump({'fine_grained_labels': label_mapping, 'attack_categories': cat_mapping}, f, indent=4)

print("Đã cập nhật: models/preprocessor.joblib & models/label_mapping.json")
print("\nPhase 2 hoàn thành xuất sắc!")